# 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

# 2. Define paths

In [2]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

CLEANED_DATA_PATH = PROCESSED_DATA_DIR / "fmcg_cleaned.csv"
MONTHLY_KPI_PATH = PROCESSED_DATA_DIR / "monthly_kpi.csv"
CATEGORY_KPI_PATH = PROCESSED_DATA_DIR / "category_kpi.csv"
CHANNEL_KPI_PATH = PROCESSED_DATA_DIR / "channel_kpi.csv"
REGION_KPI_PATH = PROCESSED_DATA_DIR / "region_kpi.csv"
COUNTRY_KPI_PATH = PROCESSED_DATA_DIR / "country_kpi.csv"
PROMOTION_KPI_PATH = PROCESSED_DATA_DIR / "promotion_kpi.csv"
PRODUCT_KPI_PATH = PROCESSED_DATA_DIR / "product_kpi.csv"

print("Processed data dir:", PROCESSED_DATA_DIR)

Processed data dir: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\processed


# 3. Load Processed Artifacts

In [3]:
df = pd.read_csv(CLEANED_DATA_PATH)
monthly_kpi = pd.read_csv(MONTHLY_KPI_PATH)
category_kpi = pd.read_csv(CATEGORY_KPI_PATH)
channel_kpi = pd.read_csv(CHANNEL_KPI_PATH)
region_kpi = pd.read_csv(REGION_KPI_PATH)
country_kpi = pd.read_csv(COUNTRY_KPI_PATH)
promotion_kpi = pd.read_csv(PROMOTION_KPI_PATH)
product_kpi = pd.read_csv(PRODUCT_KPI_PATH)

df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")

print("Cleaned data:", df.shape)
print("Monthly KPI:", monthly_kpi.shape)
print("Category KPI:", category_kpi.shape)
print("Channel KPI:", channel_kpi.shape)

Cleaned data: (18240, 52)
Monthly KPI: (36, 23)
Category KPI: (5, 19)
Channel KPI: (4, 16)


# 4. Validate P&L Fields

In [5]:
required_pnl_cols = [
    "Gross_Sales_USD",
    "Net_Revenue_USD",
    "COGS_USD",
    "Gross_Profit_USD",
    "Logistics_Cost_USD",
    "Marketing_Spend_USD",
    "Profit_USD",
    "Contribution_Profit_USD",
]

missing_pnl_cols = [col for col in required_pnl_cols if col not in df.columns]

if missing_pnl_cols:
    raise ValueError(f"Missing required P&L columns: {missing_pnl_cols}")

df[required_pnl_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Gross_Sales_USD,"18,240.00",936.43,"1,045.61",13.02,274.53,565.43,"1,213.84","10,634.12"
Net_Revenue_USD,"18,240.00",792.21,861.24,10.54,245.22,493.89,"1,027.36","8,752.94"
COGS_USD,"18,240.00",471.02,537.35,6.31,140.51,287.16,599.79,"6,473.33"
Gross_Profit_USD,"18,240.00",321.19,339.88,4.23,100.48,202.92,420.43,"3,795.49"
Logistics_Cost_USD,"18,240.00",54.03,51.27,1.08,21.37,38.98,69.24,628.86
Marketing_Spend_USD,"18,240.00",85.74,87.55,1.81,33.49,61.39,107.61,"1,767.71"
Profit_USD,"18,240.00",181.41,239.53,-637.50,32.34,91.89,239.12,"2,723.00"
Contribution_Profit_USD,"18,240.00",181.41,239.53,-637.50,32.35,91.89,239.12,"2,723.00"


# 5. Create Monthly P&L statement

In [6]:
pnl_monthly = (
    df.groupby(["Year", "Quarter", "Month", "Month_Name", "Year_Month"], as_index=False)
    .agg(
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        gross_profit=("Gross_Profit_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        contribution_profit=("Contribution_Profit_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        order_count=("Order_ID", "nunique"),
    )
)

pnl_monthly["gross_margin"] = pnl_monthly["gross_profit"] / pnl_monthly["net_revenue"].replace(0, np.nan)
pnl_monthly["contribution_margin"] = pnl_monthly["contribution_profit"] / pnl_monthly["net_revenue"].replace(0, np.nan)
pnl_monthly["profit_margin"] = pnl_monthly["profit"] / pnl_monthly["net_revenue"].replace(0, np.nan)
pnl_monthly["cogs_ratio"] = pnl_monthly["cogs"] / pnl_monthly["net_revenue"].replace(0, np.nan)
pnl_monthly["logistics_ratio"] = pnl_monthly["logistics_cost"] / pnl_monthly["net_revenue"].replace(0, np.nan)
pnl_monthly["marketing_ratio"] = pnl_monthly["marketing_spend"] / pnl_monthly["net_revenue"].replace(0, np.nan)
pnl_monthly["avg_selling_price"] = pnl_monthly["net_revenue"] / pnl_monthly["units_sold"].replace(0, np.nan)

pnl_monthly = pnl_monthly.sort_values(["Year", "Month"]).reset_index(drop=True)

pnl_monthly.head()

,Year,Quarter,Month,Month_Name,Year_Month,gross_sales,net_revenue,cogs,gross_profit,logistics_cost,marketing_spend,contribution_profit,profit,units_sold,order_count,gross_margin,contribution_margin,profit_margin,cogs_ratio,logistics_ratio,marketing_ratio,avg_selling_price
0,2023,Q1,1,January,2023-01,"481,511.88","408,301.70","242,713.37","165,588.33","27,940.41","43,286.62","94,361.30","94,361.30",110629,535,0.41,0.23,0.23,0.59,0.07,0.11,3.69
1,2023,Q1,2,February,2023-02,"419,997.02","354,938.26","210,194.58","144,743.68","24,395.52","39,437.88","80,910.28","80,910.28",98261,472,0.41,0.23,0.23,0.59,0.07,0.11,3.61
2,2023,Q1,3,March,2023-03,"438,332.68","371,116.33","222,070.30","149,046.03","24,777.09","40,046.44","84,222.50","84,222.50",105895,512,0.40,0.23,0.23,0.60,0.07,0.11,3.50
3,2023,Q2,4,April,2023-04,"421,101.02","357,347.48","212,010.72","145,336.76","25,161.47","38,509.72","81,665.57","81,665.57",100548,497,0.41,0.23,0.23,0.59,0.07,0.11,3.55
4,2023,Q2,5,May,2023-05,"427,223.81","360,818.12","216,709.26","144,108.86","24,584.69","38,239.59","81,284.58","81,284.58",99014,479,0.40,0.23,0.23,0.60,0.07,0.11,3.64


# 6. Create Category P&L Statement

In [7]:
pnl_category = (
    df.groupby("Product_Category", as_index=False)
    .agg(
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        gross_profit=("Gross_Profit_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        contribution_profit=("Contribution_Profit_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        order_count=("Order_ID", "nunique"),
    )
)

pnl_category["revenue_share"] = pnl_category["net_revenue"] / pnl_category["net_revenue"].sum()
pnl_category["profit_share"] = pnl_category["profit"] / pnl_category["profit"].sum()
pnl_category["gross_margin"] = pnl_category["gross_profit"] / pnl_category["net_revenue"].replace(0, np.nan)
pnl_category["contribution_margin"] = pnl_category["contribution_profit"] / pnl_category["net_revenue"].replace(0, np.nan)
pnl_category["profit_margin"] = pnl_category["profit"] / pnl_category["net_revenue"].replace(0, np.nan)
pnl_category["cogs_ratio"] = pnl_category["cogs"] / pnl_category["net_revenue"].replace(0, np.nan)
pnl_category["logistics_ratio"] = pnl_category["logistics_cost"] / pnl_category["net_revenue"].replace(0, np.nan)
pnl_category["marketing_ratio"] = pnl_category["marketing_spend"] / pnl_category["net_revenue"].replace(0, np.nan)

pnl_category = pnl_category.sort_values("net_revenue", ascending=False).reset_index(drop=True)

pnl_category

,Product_Category,gross_sales,net_revenue,cogs,gross_profit,logistics_cost,marketing_spend,contribution_profit,profit,units_sold,order_count,revenue_share,profit_share,gross_margin,contribution_margin,profit_margin,cogs_ratio,logistics_ratio,marketing_ratio
0,Beverages,"4,419,617.21","3,737,688.56","2,505,214.86","1,232,473.70","276,179.84","432,092.21","524,201.65","524,201.65",981268,4356,0.26,0.16,0.33,0.14,0.14,0.67,0.07,0.12
1,Household,"3,739,362.54","3,160,720.02","1,775,830.31","1,384,889.71","232,137.42","320,530.71","832,221.58","832,221.58",683008,3237,0.22,0.25,0.44,0.26,0.26,0.56,0.07,0.10
2,Personal Care,"3,266,698.71","2,765,969.05","1,523,818.75","1,242,150.30","176,580.22","288,589.92","776,980.16","776,980.16",686448,3439,0.19,0.23,0.45,0.28,0.28,0.55,0.06,0.10
3,Snacks,"2,973,481.53","2,516,605.10","1,510,797.61","1,005,807.49","158,377.39","292,218.01","555,212.09","555,212.09",912058,4283,0.17,0.17,0.40,0.22,0.22,0.60,0.06,0.12
4,Dairy & Breakfast,"2,681,355.06","2,268,937.27","1,275,821.30","993,115.97","142,264.96","230,501.18","620,349.83","620,349.83",620692,2925,0.16,0.19,0.44,0.27,0.27,0.56,0.06,0.10


# 7. Create Channel P&L Statement

In [8]:
pnl_channel = (
    df.groupby("Sales_Channel", as_index=False)
    .agg(
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        gross_profit=("Gross_Profit_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        contribution_profit=("Contribution_Profit_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        order_count=("Order_ID", "nunique"),
    )
)

pnl_channel["revenue_share"] = pnl_channel["net_revenue"] / pnl_channel["net_revenue"].sum()
pnl_channel["gross_margin"] = pnl_channel["gross_profit"] / pnl_channel["net_revenue"].replace(0, np.nan)
pnl_channel["contribution_margin"] = pnl_channel["contribution_profit"] / pnl_channel["net_revenue"].replace(0, np.nan)
pnl_channel["profit_margin"] = pnl_channel["profit"] / pnl_channel["net_revenue"].replace(0, np.nan)
pnl_channel["marketing_ratio"] = pnl_channel["marketing_spend"] / pnl_channel["net_revenue"].replace(0, np.nan)
pnl_channel["logistics_ratio"] = pnl_channel["logistics_cost"] / pnl_channel["net_revenue"].replace(0, np.nan)

pnl_channel = pnl_channel.sort_values("net_revenue", ascending=False).reset_index(drop=True)

pnl_channel

,Sales_Channel,gross_sales,net_revenue,cogs,gross_profit,logistics_cost,marketing_spend,contribution_profit,profit,units_sold,order_count,revenue_share,gross_margin,contribution_margin,profit_margin,marketing_ratio,logistics_ratio
0,Wholesale,"6,363,163.57","5,208,657.92","3,158,966.47","2,049,691.45","280,998.50","419,413.70","1,349,279.25","1,349,279.25",1456048,2848,0.36,0.39,0.26,0.26,0.08,0.05
1,Distributor,"5,293,893.40","4,451,811.56","2,620,527.07","1,831,284.49","284,669.62","423,220.89","1,123,393.98","1,123,393.98",1227016,3629,0.31,0.41,0.25,0.25,0.10,0.06
2,Modern Trade,"3,866,687.09","3,371,020.34","1,979,798.74","1,391,221.60","267,043.00","463,912.39","660,266.21","660,266.21",860610,6219,0.23,0.41,0.20,0.20,0.14,0.08
3,Online,"1,556,770.99","1,418,430.18","832,190.55","586,239.63","152,828.71","257,385.05","176,025.87","176,025.87",339800,5544,0.10,0.41,0.12,0.12,0.18,0.11


# 8. Create Region P&L Statement

In [9]:
pnl_region = (
    df.groupby("Region", as_index=False)
    .agg(
        gross_sales=("Gross_Sales_USD", "sum"),
        net_revenue=("Net_Revenue_USD", "sum"),
        cogs=("COGS_USD", "sum"),
        gross_profit=("Gross_Profit_USD", "sum"),
        logistics_cost=("Logistics_Cost_USD", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
        contribution_profit=("Contribution_Profit_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        order_count=("Order_ID", "nunique"),
    )
)

pnl_region["revenue_share"] = pnl_region["net_revenue"] / pnl_region["net_revenue"].sum()
pnl_region["gross_margin"] = pnl_region["gross_profit"] / pnl_region["net_revenue"].replace(0, np.nan)
pnl_region["contribution_margin"] = pnl_region["contribution_profit"] / pnl_region["net_revenue"].replace(0, np.nan)
pnl_region["profit_margin"] = pnl_region["profit"] / pnl_region["net_revenue"].replace(0, np.nan)
pnl_region["marketing_ratio"] = pnl_region["marketing_spend"] / pnl_region["net_revenue"].replace(0, np.nan)
pnl_region["logistics_ratio"] = pnl_region["logistics_cost"] / pnl_region["net_revenue"].replace(0, np.nan)

pnl_region = pnl_region.sort_values("net_revenue", ascending=False).reset_index(drop=True)

pnl_region

,Region,gross_sales,net_revenue,cogs,gross_profit,logistics_cost,marketing_spend,contribution_profit,profit,units_sold,order_count,revenue_share,gross_margin,contribution_margin,profit_margin,marketing_ratio,logistics_ratio
0,Europe,"5,688,917.80","4,811,497.42","2,839,662.33","1,971,835.09","312,738.43","523,676.26","1,135,420.40","1,135,420.40",1168906,5532,0.33,0.41,0.24,0.24,0.11,0.06
1,North America,"3,955,446.16","3,337,777.37","1,972,189.55","1,365,587.82","216,987.85","368,430.87","780,169.10","780,169.10",876372,4150,0.23,0.41,0.23,0.23,0.11,0.07
2,Asia,"3,478,530.30","2,946,505.41","1,757,241.35","1,189,264.06","201,231.27","317,712.33","670,320.46","670,320.46",941879,4420,0.20,0.40,0.23,0.23,0.11,0.07
3,South America,"2,044,202.18","1,728,515.24","1,020,180.19","708,335.05","124,377.39","179,731.07","404,226.59","404,226.59",510620,2303,0.12,0.41,0.23,0.23,0.10,0.07
4,Oceania,"1,913,418.61","1,625,624.56","1,002,209.41","623,415.15","130,204.89","174,381.50","318,828.76","318,828.76",385697,1835,0.11,0.38,0.20,0.20,0.11,0.08


# 9. P&L Waterfall Data

In [10]:
total_gross_sales = df["Gross_Sales_USD"].sum()
total_net_revenue = df["Net_Revenue_USD"].sum()
total_discount_impact = total_gross_sales - total_net_revenue

total_cogs = df["COGS_USD"].sum()
total_logistics = df["Logistics_Cost_USD"].sum()
total_marketing = df["Marketing_Spend_USD"].sum()
total_profit = df["Profit_USD"].sum()

pnl_waterfall = pd.DataFrame({
    "step": [
        "Gross Sales",
        "Discount / Revenue Adjustment",
        "Net Revenue",
        "COGS",
        "Logistics Cost",
        "Marketing Spend",
        "Profit",
    ],
    "amount": [
        total_gross_sales,
        -total_discount_impact,
        total_net_revenue,
        -total_cogs,
        -total_logistics,
        -total_marketing,
        total_profit,
    ],
    "type": [
        "total",
        "relative",
        "total",
        "relative",
        "relative",
        "relative",
        "total",
    ]
})

pnl_waterfall

,step,amount,type
0,Gross Sales,"17,080,515.05",total
1,Discount / Revenue Adjustment,"-2,630,595.05",relative
2,Net Revenue,"14,449,920.00",total
3,COGS,"-8,591,482.83",relative
4,Logistics Cost,"-985,539.83",relative
5,Marketing Spend,"-1,563,932.03",relative
6,Profit,"3,308,965.31",total


# 10. Visual Check: P&L Waterfall

In [11]:
fig = go.Figure(
    go.Waterfall(
        name="P&L",
        orientation="v",
        measure=pnl_waterfall["type"],
        x=pnl_waterfall["step"],
        y=pnl_waterfall["amount"],
        text=[f"{x:,.0f}" for x in pnl_waterfall["amount"]],
        textposition="outside"
    )
)

fig.update_layout(
    title="Overall P&L Waterfall",
    yaxis_title="USD",
    showlegend=False
)

fig.show()

# 11. Budget Simulation Layer

Logic:
- Budget for 2024 and 2025 = prior-year same-month actual x target growth

Assumption
- Revenue target growth = 8%
- Profit target growth = 10%
- Marketing spend target growth = 5%

2023 = Base year (actual x 1.00)

In [12]:
# Define budget assumptions

REVENUE_TARGET_GROWTH = 0.08
PROFIT_TARGET_GROWTH = 0.10
MARKETING_TARGET_GROWTH = 0.05

budget_assumptions = pd.DataFrame({
    "assumption" : [
        "Revenue target growth",
        "Profit target growth",
        "Marketing spend target growth",
        "Base year budget method"
    ],
    "value": [
        REVENUE_TARGET_GROWTH,
        PROFIT_TARGET_GROWTH,
        MARKETING_TARGET_GROWTH,
        "2023 budget equals to 2023 actual; 2024 and 2025 budget uses prior-year same-month actual x target growth"
    ]
})

budget_assumptions

,assumption,value
0,Revenue target growth,0.08
1,Profit target growth,0.10
2,Marketing spend target growth,0.05
3,Base year budget method,2023 budget equals to 2023 actual; 2024 and 20...


In [14]:
# Prepare actual monthly values

actual_monthly = pnl_monthly[[
    "Year",
    "Quarter",
    "Month",
    "Month_Name",
    "Year_Month",
    "net_revenue",
    "profit",
    "marketing_spend",
    "units_sold"
]].copy()

actual_monthly = actual_monthly.sort_values(["Year", "Month"]).reset_index(drop=True)
actual_monthly.head()

,Year,Quarter,Month,Month_Name,Year_Month,net_revenue,profit,marketing_spend,units_sold
0,2023,Q1,1,January,2023-01,"408,301.70","94,361.30","43,286.62",110629
1,2023,Q1,2,February,2023-02,"354,938.26","80,910.28","39,437.88",98261
2,2023,Q1,3,March,2023-03,"371,116.33","84,222.50","40,046.44",105895
3,2023,Q2,4,April,2023-04,"357,347.48","81,665.57","38,509.72",100548
4,2023,Q2,5,May,2023-05,"360,818.12","81,284.58","38,239.59",99014


In [15]:
# Create priot-year same-month baseline

prior_year_actual = actual_monthly[[
    "Year",
    "Month",
    "net_revenue",
    "profit",
    "marketing_spend",
    "units_sold"
]].copy()

prior_year_actual["Year"] = prior_year_actual["Year"] + 1

prior_year_actual = prior_year_actual.rename(columns={
    "net_revenue": "prior_year_net_revenue",
    "profit": "prior_year_profit",
    "marketing_spend": "prior_year_marketing_spend",
    "units_sold": "prior_year_units_sold"
})

budget_actual = actual_monthly.merge(
    prior_year_actual,
    on=["Year", "Month"],
    how="left",
)

budget_actual.head()

,Year,Quarter,Month,Month_Name,Year_Month,net_revenue,profit,marketing_spend,units_sold,prior_year_net_revenue,prior_year_profit,prior_year_marketing_spend,prior_year_units_sold
0,2023,Q1,1,January,2023-01,"408,301.70","94,361.30","43,286.62",110629,NaN,NaN,NaN,NaN
1,2023,Q1,2,February,2023-02,"354,938.26","80,910.28","39,437.88",98261,NaN,NaN,NaN,NaN
2,2023,Q1,3,March,2023-03,"371,116.33","84,222.50","40,046.44",105895,NaN,NaN,NaN,NaN
3,2023,Q2,4,April,2023-04,"357,347.48","81,665.57","38,509.72",100548,NaN,NaN,NaN,NaN
4,2023,Q2,5,May,2023-05,"360,818.12","81,284.58","38,239.59",99014,NaN,NaN,NaN,NaN


In [16]:
# Create budget values

base_year = budget_actual["Year"].min()

budget_actual["budget_net_revenue"] = np.where(
    budget_actual["prior_year_net_revenue"].notna(),
    budget_actual["prior_year_net_revenue"] * (1 + REVENUE_TARGET_GROWTH),
    budget_actual["net_revenue"]
)

budget_actual["budget_profit"] = np.where(
    budget_actual["prior_year_profit"].notna(),
    budget_actual["prior_year_profit"] * (1 + PROFIT_TARGET_GROWTH),
    budget_actual["profit"]
)

budget_actual["budget_marketing_spend"] = np.where(
    budget_actual["prior_year_marketing_spend"].notna(),
    budget_actual["prior_year_marketing_spend"] * (1 + MARKETING_TARGET_GROWTH),
    budget_actual["marketing_spend"]
)

budget_actual["budget_units_sold"] = np.where(
    budget_actual["prior_year_units_sold"].notna(),
    budget_actual["prior_year_units_sold"] * (1 + REVENUE_TARGET_GROWTH),
    budget_actual["units_sold"]
)

budget_actual.head()

,Year,Quarter,Month,Month_Name,Year_Month,net_revenue,profit,marketing_spend,units_sold,prior_year_net_revenue,prior_year_profit,prior_year_marketing_spend,prior_year_units_sold,budget_net_revenue,budget_profit,budget_marketing_spend,budget_units_sold
0,2023,Q1,1,January,2023-01,"408,301.70","94,361.30","43,286.62",110629,NaN,NaN,NaN,NaN,"408,301.70","94,361.30","43,286.62","110,629.00"
1,2023,Q1,2,February,2023-02,"354,938.26","80,910.28","39,437.88",98261,NaN,NaN,NaN,NaN,"354,938.26","80,910.28","39,437.88","98,261.00"
2,2023,Q1,3,March,2023-03,"371,116.33","84,222.50","40,046.44",105895,NaN,NaN,NaN,NaN,"371,116.33","84,222.50","40,046.44","105,895.00"
3,2023,Q2,4,April,2023-04,"357,347.48","81,665.57","38,509.72",100548,NaN,NaN,NaN,NaN,"357,347.48","81,665.57","38,509.72","100,548.00"
4,2023,Q2,5,May,2023-05,"360,818.12","81,284.58","38,239.59",99014,NaN,NaN,NaN,NaN,"360,818.12","81,284.58","38,239.59","99,014.00"


In [17]:
# Calculate variance

budget_actual["revenue_variance"] = budget_actual["net_revenue"] - budget_actual["budget_net_revenue"]
budget_actual["profit_variance"] = budget_actual["profit"] - budget_actual["budget_profit"]
budget_actual["marketing_spend_variance"] = budget_actual["marketing_spend"] - budget_actual["budget_marketing_spend"]
budget_actual["units_variance"] = budget_actual["units_sold"] - budget_actual["budget_units_sold"]

budget_actual["revenue_variance_pct"] = budget_actual["revenue_variance"] / budget_actual["budget_net_revenue"].replace(0, np.nan)
budget_actual["profit_variance_pct"] = budget_actual["profit_variance"] / budget_actual["budget_profit"].replace(0, np.nan)
budget_actual["marketing_spend_variance_pct"] = budget_actual["marketing_spend_variance"] / budget_actual["budget_marketing_spend"].replace(0, np.nan)
budget_actual["units_variance_pct"] = budget_actual["units_variance"] / budget_actual["budget_units_sold"].replace(0, np.nan)

budget_actual["revenue_budget_achievement"] = budget_actual["net_revenue"] / budget_actual["budget_net_revenue"].replace(0, np.nan)
budget_actual["profit_budget_achievement"] = budget_actual["profit"] / budget_actual["budget_profit"].replace(0, np.nan)

budget_actual["budget_status"] = np.where(
    budget_actual["revenue_variance"] >= 0,
    "Above Budget",
    "Below Budget"
)

budget_actual.head()

,Year,Quarter,Month,Month_Name,Year_Month,net_revenue,profit,marketing_spend,units_sold,prior_year_net_revenue,prior_year_profit,prior_year_marketing_spend,prior_year_units_sold,budget_net_revenue,budget_profit,budget_marketing_spend,budget_units_sold,revenue_variance,profit_variance,marketing_spend_variance,units_variance,revenue_variance_pct,profit_variance_pct,marketing_spend_variance_pct,units_variance_pct,revenue_budget_achievement,profit_budget_achievement,budget_status
0,2023,Q1,1,January,2023-01,"408,301.70","94,361.30","43,286.62",110629,NaN,NaN,NaN,NaN,"408,301.70","94,361.30","43,286.62","110,629.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,Above Budget
1,2023,Q1,2,February,2023-02,"354,938.26","80,910.28","39,437.88",98261,NaN,NaN,NaN,NaN,"354,938.26","80,910.28","39,437.88","98,261.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,Above Budget
2,2023,Q1,3,March,2023-03,"371,116.33","84,222.50","40,046.44",105895,NaN,NaN,NaN,NaN,"371,116.33","84,222.50","40,046.44","105,895.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,Above Budget
3,2023,Q2,4,April,2023-04,"357,347.48","81,665.57","38,509.72",100548,NaN,NaN,NaN,NaN,"357,347.48","81,665.57","38,509.72","100,548.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,Above Budget
4,2023,Q2,5,May,2023-05,"360,818.12","81,284.58","38,239.59",99014,NaN,NaN,NaN,NaN,"360,818.12","81,284.58","38,239.59","99,014.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,1.00,Above Budget


In [18]:
# Budget summary by year

budget_summary_year = (
    budget_actual.groupby("Year", as_index=False)
    .agg(
        actual_net_revenue=("net_revenue", "sum"),
        budget_net_revenue=("budget_net_revenue", "sum"),
        actual_profit=("profit", "sum"),
        budget_profit=("budget_profit", "sum"),
        actual_marketing_spend=("marketing_spend", "sum"),
        budget_marketing_spend=("budget_marketing_spend", "sum"),
        actual_units_sold=("units_sold", "sum"),
        budget_units_sold=("budget_units_sold", "sum"),
    )
)

budget_summary_year["revenue_variance"] = budget_summary_year["actual_net_revenue"] - budget_summary_year["budget_net_revenue"]
budget_summary_year["profit_variance"] = budget_summary_year["actual_profit"] - budget_summary_year["budget_profit"]
budget_summary_year["revenue_budget_achievement"] = budget_summary_year["actual_net_revenue"] / budget_summary_year["budget_net_revenue"].replace(0, np.nan)
budget_summary_year["profit_budget_achievement"] = budget_summary_year["actual_profit"] / budget_summary_year["budget_profit"].replace(0, np.nan)

budget_summary_year

,Year,actual_net_revenue,budget_net_revenue,actual_profit,budget_profit,actual_marketing_spend,budget_marketing_spend,actual_units_sold,budget_units_sold,revenue_variance,profit_variance,revenue_budget_achievement,profit_budget_achievement
0,2023,"4,593,306.60","4,593,306.60","1,033,484.00","1,033,484.00","502,131.92","502,131.92",1265124,"1,265,124.00",0.00,0.00,1.00,1.00
1,2024,"4,840,333.97","4,960,771.13","1,114,353.73","1,136,832.40","519,667.19","527,238.52",1312628,"1,366,333.92","-120,437.16","-22,478.67",0.98,0.98
2,2025,"5,016,279.43","5,227,560.69","1,161,127.58","1,225,789.10","542,132.92","545,650.55",1305722,"1,417,638.24","-211,281.26","-64,661.52",0.96,0.95


In [19]:
# Actual vs Budget revenue plot

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=budget_actual["Year_Month"],
        y=budget_actual["net_revenue"],
        mode="lines+markers",
        name="Actual Net Revenue"
    )
)

fig.add_trace(
    go.Scatter(
        x=budget_actual["Year_Month"],
        y=budget_actual["budget_net_revenue"],
        mode="lines+markers",
        name="Budget Net Revenue"
    )
)

fig.update_layout(
    title="Actual vs Budget Net Revenue",
    xaxis_title="Month",
    yaxis_title="USD"
)

fig.show()

In [20]:
# Create category budget variance

category_monthly_actual = (
    df.groupby(["Year", "Quarter", "Month", "Month_Name", "Year_Month", "Product_Category"], as_index=False)
    .agg(
        net_revenue=("Net_Revenue_USD", "sum"),
        profit=("Profit_USD", "sum"),
        units_sold=("Units_Sold", "sum"),
        marketing_spend=("Marketing_Spend_USD", "sum"),
    )
)

category_prior_year = category_monthly_actual[[
    "Year",
    "Month",
    "Product_Category",
    "net_revenue",
    "profit",
    "units_sold",
    "marketing_spend",
]].copy()

category_prior_year["Year"] = category_prior_year["Year"] + 1

category_prior_year = category_prior_year.rename(columns={
    "net_revenue": "prior_year_net_revenue",
    "profit": "prior_year_profit",
    "units_sold": "prior_year_units_sold",
    "marketing_spend": "prior_year_marketing_spend",
})

category_budget_actual = category_monthly_actual.merge(
    category_prior_year,
    on=["Year", "Month", "Product_Category"],
    how="left",
)

category_budget_actual["budget_net_revenue"] = np.where(
    category_budget_actual["prior_year_net_revenue"].notna(),
    category_budget_actual["prior_year_net_revenue"] * (1 + REVENUE_TARGET_GROWTH),
    category_budget_actual["net_revenue"]
)

category_budget_actual["budget_profit"] = np.where(
    category_budget_actual["prior_year_profit"].notna(),
    category_budget_actual["prior_year_profit"] * (1 + PROFIT_TARGET_GROWTH),
    category_budget_actual["profit"]
)

category_budget_actual["revenue_variance"] = (
    category_budget_actual["net_revenue"] - category_budget_actual["budget_net_revenue"]
)
category_budget_actual["profit_variance"] = (
    category_budget_actual["profit"] - category_budget_actual["budget_profit"]
)

category_budget_actual["revenue_variance_pct"] = (
    category_budget_actual["revenue_variance"] / category_budget_actual["budget_net_revenue"].replace(0, np.nan)
)
category_budget_actual["profit_variance_pct"] = (
    category_budget_actual["profit_variance"] / category_budget_actual["budget_profit"].replace(0, np.nan)
)

category_budget_actual["budget_status"] = np.where(
    category_budget_actual["revenue_variance"] >= 0,
    "Above Budget",
    "Below Budget"
)

category_budget_actual.head()

,Year,Quarter,Month,Month_Name,Year_Month,Product_Category,net_revenue,profit,units_sold,marketing_spend,prior_year_net_revenue,prior_year_profit,prior_year_units_sold,prior_year_marketing_spend,budget_net_revenue,budget_profit,revenue_variance,profit_variance,revenue_variance_pct,profit_variance_pct,budget_status
0,2023,Q1,1,January,2023-01,Beverages,"103,920.29","15,812.53",28790,"11,502.37",NaN,NaN,NaN,NaN,"103,920.29","15,812.53",0.00,0.00,0.00,0.00,Above Budget
1,2023,Q1,1,January,2023-01,Dairy & Breakfast,"68,371.87","18,388.37",18470,"6,980.53",NaN,NaN,NaN,NaN,"68,371.87","18,388.37",0.00,0.00,0.00,0.00,Above Budget
2,2023,Q1,1,January,2023-01,Household,"86,874.99","22,847.98",19717,"9,071.50",NaN,NaN,NaN,NaN,"86,874.99","22,847.98",0.00,0.00,0.00,0.00,Above Budget
3,2023,Q1,1,January,2023-01,Personal Care,"81,563.72","23,354.68",19216,"7,700.70",NaN,NaN,NaN,NaN,"81,563.72","23,354.68",0.00,0.00,0.00,0.00,Above Budget
4,2023,Q1,1,January,2023-01,Snacks,"67,570.83","13,957.74",24436,"8,031.52",NaN,NaN,NaN,NaN,"67,570.83","13,957.74",0.00,0.00,0.00,0.00,Above Budget


In [21]:
# Category budget summary

category_budget_summary = (
    category_budget_actual.groupby("Product_Category", as_index=False)
    .agg(
        actual_net_revenue=("net_revenue", "sum"),
        budget_net_revenue=("budget_net_revenue", "sum"),
        actual_profit=("profit", "sum"),
        budget_profit=("budget_profit", "sum"),
        below_budget_months=("budget_status", lambda x: (x == "Below Budget").sum()),
        total_months=("budget_status", "count"),
    )
)

category_budget_summary["revenue_variance"] = (
    category_budget_summary["actual_net_revenue"] - category_budget_summary["budget_net_revenue"]
)
category_budget_summary["profit_variance"] = (
    category_budget_summary["actual_profit"] - category_budget_summary["budget_profit"]
)
category_budget_summary["revenue_budget_achievement"] = (
    category_budget_summary["actual_net_revenue"] / category_budget_summary["budget_net_revenue"].replace(0, np.nan)
)
category_budget_summary["profit_budget_achievement"] = (
    category_budget_summary["actual_profit"] / category_budget_summary["budget_profit"].replace(0, np.nan)
)
category_budget_summary["below_budget_rate"] = (
    category_budget_summary["below_budget_months"] / category_budget_summary["total_months"].replace(0, np.nan)
)

category_budget_summary = category_budget_summary.sort_values(
    "revenue_variance",
    ascending=True
).reset_index(drop=True)

category_budget_summary

,Product_Category,actual_net_revenue,budget_net_revenue,actual_profit,budget_profit,below_budget_months,total_months,revenue_variance,profit_variance,revenue_budget_achievement,profit_budget_achievement,below_budget_rate
0,Beverages,"3,737,688.56","3,984,760.89","524,201.65","563,492.03",16,36,"-247,072.33","-39,290.38",0.94,0.93,0.44
1,Snacks,"2,516,605.10","2,642,510.90","555,212.09","590,817.95",14,36,"-125,905.80","-35,605.86",0.95,0.94,0.39
2,Household,"3,160,720.02","3,172,340.38","832,221.58","842,717.81",14,36,"-11,620.36","-10,496.23",1.00,0.99,0.39
3,Dairy & Breakfast,"2,268,937.27","2,256,388.39","620,349.83","623,005.40",13,36,"12,548.88","-2,655.57",1.01,1.00,0.36
4,Personal Care,"2,765,969.05","2,725,637.86","776,980.16","776,072.31",12,36,"40,331.19",907.85,1.01,1.00,0.33


# 12. Business Insight

In [22]:
latest_year = budget_actual["Year"].max()

latest_year_summary = budget_summary_year[
    budget_summary_year["Year"] == latest_year
].iloc[0]

worst_category = category_budget_summary.iloc[0]["Product_Category"]
best_revenue_category = pnl_category.sort_values("net_revenue", ascending=False).iloc[0]["Product_Category"]
best_margin_category = pnl_category.sort_values("profit_margin", ascending=False).iloc[0]["Product_Category"]

business_insights = pd.DataFrame({
    "insight_type": [
        "Latest year revenue achievement",
        "Latest year profit achievement",
        "Largest negative category variance",
        "Top revenue category",
        "Highest margin category",
        "Negative profit transaction count",
    ],
    "insight": [
        f"{latest_year} revenue achievement reached {latest_year_summary['revenue_budget_achievement']:.1%}.",
        f"{latest_year} profit achievement reached {latest_year_summary['profit_budget_achievement']:.1%}.",
        f"{worst_category} had the largest negative revenue variance against simulated budget.",
        f"{best_revenue_category} generated the highest net revenue.",
        f"{best_margin_category} delivered the highest profit margin.",
        f"{int(df['is_negative_profit'].sum()):,} transactions generated negative profit and should be reviewed for pricing, promotion, or cost pressure.",
    ]
})

business_insights

,insight_type,insight
0,Latest year revenue achievement,2025 revenue achievement reached 96.0%.
1,Latest year profit achievement,2025 profit achievement reached 94.7%.
2,Largest negative category variance,Beverages had the largest negative revenue var...
3,Top revenue category,Beverages generated the highest net revenue.
4,Highest margin category,Personal Care delivered the highest profit mar...
5,Negative profit transaction count,784 transactions generated negative profit and...


# 13. Save P&L and Budget Artifacts

In [23]:
pnl_monthly.to_csv(PROCESSED_DATA_DIR / "pnl_monthly.csv", index=False)
pnl_category.to_csv(PROCESSED_DATA_DIR / "pnl_category.csv", index=False)
pnl_channel.to_csv(PROCESSED_DATA_DIR / "pnl_channel.csv", index=False)
pnl_region.to_csv(PROCESSED_DATA_DIR / "pnl_region.csv", index=False)
pnl_waterfall.to_csv(PROCESSED_DATA_DIR / "pnl_waterfall.csv", index=False)

budget_actual.to_csv(PROCESSED_DATA_DIR / "budget_actual.csv", index=False)
budget_summary_year.to_csv(PROCESSED_DATA_DIR / "budget_summary_year.csv", index=False)
category_budget_actual.to_csv(PROCESSED_DATA_DIR / "category_budget_actual.csv", index=False)
category_budget_summary.to_csv(PROCESSED_DATA_DIR / "category_budget_summary.csv", index=False)

budget_assumptions.to_csv(PROCESSED_DATA_DIR / "budget_assumptions.csv", index=False)
business_insights.to_csv(PROCESSED_DATA_DIR / "business_insights.csv", index=False)

print("Saved P&L and budget artifacts to:", PROCESSED_DATA_DIR)

Saved P&L and budget artifacts to: c:\Users\Anwar\Documents\Work duty\fmcg-business-performance-analytics_new\data\processed


In [25]:
saved_files = [
    "pnl_monthly.csv",
    "pnl_category.csv",
    "pnl_channel.csv",
    "pnl_region.csv",
    "pnl_waterfall.csv",
    "budget_actual.csv",
    "budget_summary_year.csv",
    "category_budget_actual.csv",
    "category_budget_summary.csv",
    "budget_assumptions.csv",
    "business_insights.csv",
]

for file in saved_files:
    path = PROCESSED_DATA_DIR / file
    temp = pd.read_csv(path)
    print(f"{file}: {temp.shape[0]:,} rows × {temp.shape[1]:,} columns")

pnl_monthly.csv: 36 rows × 22 columns
pnl_category.csv: 5 rows × 19 columns
pnl_channel.csv: 4 rows × 17 columns
pnl_region.csv: 5 rows × 17 columns
pnl_waterfall.csv: 7 rows × 3 columns
budget_actual.csv: 36 rows × 28 columns
budget_summary_year.csv: 3 rows × 13 columns
category_budget_actual.csv: 180 rows × 21 columns
category_budget_summary.csv: 5 rows × 12 columns
budget_assumptions.csv: 4 rows × 2 columns
business_insights.csv: 6 rows × 2 columns
